In [2]:
import matplotlib.pyplot as plt
import numpy as np
import math as maths
import torch, torch.nn as nn
import pandas as pd
import csv

torch.cuda.is_available()

True

In [3]:
data = pd.read_csv('hly532.csv',skiprows=41, low_memory=False)
data = data.rename(columns={'ind': 'Rainfall_indicator', "ind.1": "Dry_bulb_indicator", "ind.2": "Wet_bulb_indicator", "ind.3": "wind_speed_indicator", "ind.4": "wind_dir_indicator"})

# these look numeric but are stored as strings because missing readings are
# a literal " " rather than blank/NaN -- coerce to numeric, turning those
# blanks into real NaNs, then forward/back-fill them
numeric_like_cols = ["vappr", "rhum", "wddir", "vis", "clht", "clamt"]
data[numeric_like_cols] = data[numeric_like_cols].apply(pd.to_numeric, errors="coerce")
data[numeric_like_cols] = data[numeric_like_cols].ffill().bfill()

data_check = data.isnull().sum()

data_time = [int(data[-5:-3]) for data in data["date"]]
data["date"] = data_time
data.rename(columns={"date": "time"}, inplace=True)


data = data[:-1* (data.shape[0] % 24)]

In [4]:
print(data.shape)

(707112, 21)


In [5]:
print(data.head())


   time  Rainfall_indicator  rain  Dry_bulb_indicator  temp  \
0     0                   0   0.0                   0   7.8   
1     1                   0   0.6                   0   7.2   
2     2                   0   0.1                   0   7.2   
3     3                   0   0.0                   0   7.1   
4     4                   2   0.0                   0   6.6   

   Wet_bulb_indicator  wetb  dewpt  vappr  rhum  ...  wind_speed_indicator  \
0                   0   5.8    3.3    7.6  72.0  ...                     1   
1                   0   6.0    4.4    8.4  83.0  ...                     1   
2                   0   6.0    4.4    8.3  82.0  ...                     1   
3                   0   5.5    3.3    7.8  77.0  ...                     1   
4                   0   5.1    2.7    7.5  77.0  ...                     1   

   wdsp  wind_dir_indicator  wddir  ww  w  sun      vis  clht  clamt  
0    19                   1  160.0   2  6  0.0  10000.0  19.0    8.0  
1    20   

In [6]:
print(data_check)

date                    0
Rainfall_indicator      0
rain                    0
Dry_bulb_indicator      0
temp                    0
Wet_bulb_indicator      0
wetb                    0
dewpt                   0
vappr                   0
rhum                    0
msl                     0
wind_speed_indicator    0
wdsp                    0
wind_dir_indicator      0
wddir                   0
ww                      0
w                       0
sun                     0
vis                     0
clht                    0
clamt                   0
dtype: int64


In [7]:
target = data["temp"]
data = data.drop(columns=["temp"])

print(data.head())

   time  Rainfall_indicator  rain  Dry_bulb_indicator  Wet_bulb_indicator  \
0     0                   0   0.0                   0                   0   
1     1                   0   0.6                   0                   0   
2     2                   0   0.1                   0                   0   
3     3                   0   0.0                   0                   0   
4     4                   2   0.0                   0                   0   

   wetb  dewpt  vappr  rhum     msl  wind_speed_indicator  wdsp  \
0   5.8    3.3    7.6  72.0  1015.3                     1    19   
1   6.0    4.4    8.4  83.0  1015.2                     1    20   
2   6.0    4.4    8.3  82.0  1015.2                     1    20   
3   5.5    3.3    7.8  77.0  1015.4                     1    21   
4   5.1    2.7    7.5  77.0  1015.4                     1    22   

   wind_dir_indicator  wddir  ww  w  sun      vis  clht  clamt  
0                   1  160.0   2  6  0.0  10000.0  19.0    8.0  
1   

In [8]:
print("Data shape:", data.shape)

data_train_split = int(data.shape[0]/24 * 0.7)
data_validate_split = int(data.shape[0]/24 * 0.85)
data_test_split = int(data.shape[0]/24)

data_train = data[:data_train_split*24]
data_validate = data[data_train_split*24:data_validate_split*24]
data_test = data[data_validate_split*24:data_test_split*24]
print("Train shape:", data_train.shape)
print("Validate shape:", data_validate.shape)
print("Test shape:", data_test.shape)
print("Data split successfully!" if data_train.shape[0] + data_validate.shape[0] + data_test.shape[0] == data.shape[0] else "Data split failed!")


Data shape: (707112, 20)
Train shape: (494976, 20)
Validate shape: (106056, 20)
Test shape: (106080, 20)
Data split successfully!


In [9]:
def _conv1d_out_len(length, layer):
    """Output length of a Conv1d layer applied to a sequence of the given length."""
    return (length + 2 * layer.padding[0] - layer.dilation[0] * (layer.kernel_size[0] - 1) - 1) // layer.stride[0] + 1


class SimpleCNN(nn.Module):
    def __init__(self, input_size, seq_len, output_size=1):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels=input_size, out_channels=4, kernel_size=3)
        self.bn1 = nn.BatchNorm1d(4)

        self.conv2 = nn.Conv1d(in_channels=4, out_channels=16, kernel_size=3, stride=2)
        self.bn2 = nn.BatchNorm1d(16)

        out_len = _conv1d_out_len(seq_len, self.conv1)
        out_len = _conv1d_out_len(out_len, self.conv2)
        flat_dim = 16 * out_len

        self.fc1 = nn.Linear(flat_dim, 128)
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(128, output_size)

    def forward(self, x):
        # x shape: (batch, input_size, seq_len)
        x = nn.functional.gelu(self.bn1(self.conv1(x)))
        x = nn.functional.gelu(self.bn2(self.conv2(x)))
        x = x.view(x.size(0), -1)  # Flatten the tensor
        x = nn.functional.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x.squeeze(-1)  # (batch,) so it matches a size-1 target


# Example: 20 feature columns (excluding 'time'), 24-hour sequences (one day per sample)
num_features = data.shape[1] - 1  # -1 for the 'time' column
seq_len = 24

model = SimpleCNN(input_size=num_features, seq_len=seq_len, output_size=1)
print(model)

# To build a batch of inputs from a day-chunked split (e.g. data_train), drop 'time'
# (now an int 0-23, so it converts to a tensor fine on its own -- dropped here only
# because within a fixed 24-hour chunk it's a constant sequence [0..23] on every
# sample, so it carries no information the model can use), then reshape
# from (n_days*24, num_features) to (n_days, num_features, seq_len):
#
# x = torch.tensor(data_train.drop(columns=["time"]).values, dtype=torch.float32)
# x = x.view(-1, seq_len, num_features).permute(0, 2, 1)  # (n_days, num_features, seq_len)

SimpleCNN(
  (conv1): Conv1d(19, 4, kernel_size=(3,), stride=(1,))
  (bn1): BatchNorm1d(4, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (conv2): Conv1d(4, 16, kernel_size=(3,), stride=(2,))
  (bn2): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (fc1): Linear(in_features=160, out_features=128, bias=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc2): Linear(in_features=128, out_features=1, bias=True)
)


In [10]:
from torch.utils.data import Dataset, DataLoader

target_train    = target[:data_train_split*24]
target_validate = target[data_train_split*24:data_validate_split*24]
target_test     = target[data_validate_split*24:data_test_split*24]

feature_cols = [c for c in data.columns if c != "time"]

# ============================================================
# STANDARDIZATION -- fit on TRAIN ONLY, applied to TRAIN + VALIDATE ONLY.
# `data_test` is left untouched (raw) on purpose: see the folding step
# in the test cell below, which bakes this exact transform into the
# model instead of applying it to the test data.
# ============================================================
feature_mean = data_train[feature_cols].mean()
feature_std  = data_train[feature_cols].std().replace(0, 1)  # guard against constant columns

def scale_features(df):
    df = df.copy()
    df[feature_cols] = (df[feature_cols] - feature_mean) / feature_std
    return df

data_train_scaled    = scale_features(data_train)      # standardized
data_validate_scaled = scale_features(data_validate)    # standardized
# data_test is NOT scaled here -- used raw further down
# ============================================================


def make_day_tensors(df, target_series, reduce="mean"):
    """
    df: a day-chunked slice of `data` (rows = n_days * seq_len)
    target_series: the matching slice of `target`
    reduce: how to collapse each day's 24 target values into one scalar
            -- change this if your target should be something other than
            the daily mean, e.g. "last" for the final hour of the day
    """
    n_days = df.shape[0] // seq_len

    X = torch.tensor(df.drop(columns=["time"]).values, dtype=torch.float32)
    X = X.view(n_days, seq_len, num_features).permute(0, 2, 1)  # (n_days, num_features, seq_len)

    y_raw = torch.tensor(target_series.values, dtype=torch.float32).view(n_days, seq_len)
    if reduce == "mean":
        y = y_raw.mean(dim=1)
    elif reduce == "last":
        y = y_raw[:, -1]
    elif reduce == "max":
        y = y_raw.max(dim=1).values
    elif reduce == "min":
        y = y_raw.min(dim=1).values
    else:
        raise ValueError(f"Unknown reduce: {reduce}")

    return X, y


class WeatherDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


X_train, y_train         = make_day_tensors(data_train_scaled, target_train)       # standardized inputs
X_validate, y_validate   = make_day_tensors(data_validate_scaled, target_validate)  # standardized inputs
X_test, y_test           = make_day_tensors(data_test, target_test)                # RAW, unstandardized inputs

train_loader    = DataLoader(WeatherDataset(X_train, y_train), batch_size=64, shuffle=True)
validate_loader = DataLoader(WeatherDataset(X_validate, y_validate), batch_size=64, shuffle=False)
test_loader     = DataLoader(WeatherDataset(X_test, y_test), batch_size=64, shuffle=False)

print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_validate: {X_validate.shape}, y_validate: {y_validate.shape}")
print(f"X_test: {X_test.shape}, y_test: {y_test.shape}")


X_train: torch.Size([20624, 19, 24]), y_train: torch.Size([20624])
X_validate: torch.Size([4419, 19, 24]), y_validate: torch.Size([4419])
X_test: torch.Size([4420, 19, 24]), y_test: torch.Size([4420])


In [11]:
import copy

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SimpleCNN(input_size=num_features, seq_len=seq_len, output_size=1).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)

num_epochs = 50
best_val_loss = float("inf")
best_state = None

for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()
        preds = model(X_batch)
        loss = criterion(preds, y_batch)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * X_batch.size(0)
    train_loss /= len(train_loader.dataset)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for X_batch, y_batch in validate_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            preds = model(X_batch)
            loss = criterion(preds, y_batch)
            val_loss += loss.item() * X_batch.size(0)
    val_loss /= len(validate_loader.dataset)

    scheduler.step(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = copy.deepcopy(model.state_dict())

    print(f"Epoch {epoch+1}/{num_epochs} - train_loss: {train_loss:.4f} - val_loss: {val_loss:.4f}")

model.load_state_dict(best_state)
print(f"Restored best model, val_loss: {best_val_loss:.4f}")

Epoch 1/50 - train_loss: 10.5592 - val_loss: 1.7465
Epoch 2/50 - train_loss: 2.1484 - val_loss: 1.0459
Epoch 3/50 - train_loss: 1.8236 - val_loss: 1.2639
Epoch 4/50 - train_loss: 1.6575 - val_loss: 0.7040
Epoch 5/50 - train_loss: 1.4812 - val_loss: 0.5422
Epoch 6/50 - train_loss: 1.5592 - val_loss: 0.4693
Epoch 7/50 - train_loss: 1.3759 - val_loss: 0.7971
Epoch 8/50 - train_loss: 1.3278 - val_loss: 0.3652
Epoch 9/50 - train_loss: 1.2964 - val_loss: 0.3831
Epoch 10/50 - train_loss: 1.2040 - val_loss: 0.5654
Epoch 11/50 - train_loss: 1.2633 - val_loss: 0.4514
Epoch 12/50 - train_loss: 1.1649 - val_loss: 0.2294
Epoch 13/50 - train_loss: 1.1281 - val_loss: 0.1881
Epoch 14/50 - train_loss: 1.1921 - val_loss: 0.4045
Epoch 15/50 - train_loss: 1.1579 - val_loss: 0.1684
Epoch 16/50 - train_loss: 1.1324 - val_loss: 0.4095
Epoch 17/50 - train_loss: 1.1609 - val_loss: 0.2197
Epoch 18/50 - train_loss: 1.1646 - val_loss: 0.2307
Epoch 19/50 - train_loss: 1.1017 - val_loss: 0.1787
Epoch 20/50 - train_

In [12]:
import copy

# ============================================================
# SCALE THE MODEL BACK -- fold standardization into conv1 instead of
# scaling data_test.
#
# The model was trained on standardized inputs: x_scaled = (x - mean) / std.
# Conv1d is linear per input channel, so that transform can be absorbed
# directly into conv1's weight and bias, producing a model that takes
# RAW features and gives IDENTICAL outputs to the standardized model:
#
#   new_weight[:, c, :] = weight[:, c, :] / std[c]
#   new_bias            = bias - sum_{c,k} weight[:, c, k] * mean[c] / std[c]
#
# i.e. instead of "un-standardizing" the test data, we "un-standardize"
# the model so it already expects raw input.
# ============================================================
model_raw = copy.deepcopy(model)  # keep the standardized-input `model` untouched

mean = torch.tensor(feature_mean.values, dtype=torch.float32, device=device)
std  = torch.tensor(feature_std.values, dtype=torch.float32, device=device)

with torch.no_grad():
    inv_std = 1.0 / std
    w = model_raw.conv1.weight  # (out_channels, in_channels, kernel_size)
    b = model_raw.conv1.bias    # (out_channels,)

    correction = (w * mean.view(1, -1, 1) * inv_std.view(1, -1, 1)).sum(dim=(1, 2))
    w.mul_(inv_std.view(1, -1, 1))   # <- weights rescaled back to raw-input space
    b.sub_(correction)               # <- bias corrected to match
# ============================================================

model_raw.eval()
test_loss = 0.0
with torch.no_grad():
    for X_batch, y_batch in test_loader:  # X_batch is raw, unstandardized
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        preds = model_raw(X_batch)
        loss = criterion(preds, y_batch)
        test_loss += loss.item() * X_batch.size(0)
test_loss /= len(test_loader.dataset)
print(f"Test MSE (raw, unstandardized inputs): {test_loss:.4f}")

Test MSE (raw, unstandardized inputs): 0.0490


In [13]:
print("Target variance (daily mean temp -- what the model is actually trained/scored on):")
print(f"  train:    var={y_train.var():.4f}, std={y_train.std():.4f}, mean={y_train.mean():.4f}, n={len(y_train)}")
print(f"  validate: var={y_validate.var():.4f}, std={y_validate.std():.4f}, mean={y_validate.mean():.4f}, n={len(y_validate)}")
print(f"  test:     var={y_test.var():.4f}, std={y_test.std():.4f}, mean={y_test.mean():.4f}, n={len(y_test)}")


Target variance (daily mean temp -- what the model is actually trained/scored on):
  train:    var=19.2243, std=4.3845, mean=9.5366, n=20624
  validate: var=20.3866, std=4.5151, mean=9.7817, n=4419
  test:     var=19.9306, std=4.4644, mean=10.1944, n=4420


# Persistence test

In [17]:
print(target_test.values)

[12.6 11.8 12.2 ... 14.7 14.3 14.1]


In [30]:
#preds from persistence -- use the daily-mean target (y_test) the model predicts,
# not the hourly target_test, so all arrays line up (one value per day)

y_test_np = y_test.numpy()

persistence_preds = y_test_np[:-1]    # today's mean predicts tomorrow's
target_test_values = y_test_np[1:]

model_raw.eval()
with torch.no_grad():
    model_preds = model_raw(X_test.to(device)).cpu().numpy()[1:]  # drop day 0 to cover the same days

print(f"Persistence predictions: {persistence_preds.shape} ...")
print(f"Model predictions: {model_preds.shape} ...")
print(f"Target test values: {target_test_values.shape} ...")

persistence_performance = np.mean((target_test_values - persistence_preds) ** 2)
model_performance = np.mean((target_test_values - model_preds) ** 2)

print(f"Persistence MSE: {persistence_performance:.4f}")
print(f"Model MSE: {model_performance:.4f}")


Persistence predictions: (4419,) ...
Model predictions: (4419,) ...
Target test values: (4419,) ...
Persistence MSE: 4.1660
Model MSE: 0.0490
